In [1]:

import sys,os
from datetime import datetime
from pathlib import Path
import json
import torch # 张量、自动求导等
import torch.nn as nn # 网络层，Conv1D、LSTM等
import torch.nn.functional as F # 函数层， ReLU、cross_entropy等
from pyexpat import features
from torchinfo import summary
from torch.utils.data import DataLoader,TensorDataset
import time
import gc
from sklearn.metrics import roc_auc_score
import re
import math
import mne
from zmq import DEALER

from train import learning_rate, hidden_size, channal_attention, temporal_attention
import numpy as np

root = Path(r"C:\Users\hank\Desktop\Research Reproduction\dianxian\self")
chb_data_path = Path(r"D:\data\dianxian\channal_select\chb03")

if str(root) not in sys.path:
    sys.path.append(str(root))

import tools as t
import models as m

log_dir = root / "train_log"
embeddings = root / 'embeddings'
weight = root / 'weight'
Fp1_F7 = embeddings / 'Fp1-F7'
Fp2_F8 = embeddings / 'Fp2-F8'
T7_P7 = embeddings / 'T7-P7'
T8_P8 = embeddings / 'T8-P8'
P7_O1 = embeddings / 'P7-O1'
P8_O2 = embeddings / 'P8-O2'

seed = 1
val = (0.2,0.1)
method = "LOOCV"
out_channals = 32
batch = 32
elec = 6
embed_dim = 32
num_heads = 2
window_length = 10 * 256
window_overlap = 5 * 256
elec_names = ['Fp1-F7', 'Fp2-F8', 'T7-P7', 'T8-P8', 'P7-O1', 'P8-O2']

learning_rate = 0.0001
weight_decay = 0.001
channal_attention = False
temporal_attention = False
batch_size = 32
dropout = 0.5
patience = 12
epochs = 100
hidden_size = 20

In [2]:
def train_elec_model(model, optimizer, loss_function, schedule, dataset, elec_idx, elec_names, weight_dir,
                     device="cuda", patience=5, epochs=30, batch_size=32):
    """训练单电极模型，返回 (权重路径列表, 日志字典)"""
    model.to(device)
    init_state = {k: v.clone() for k, v in model.state_dict().items()}
    now = datetime.now().strftime("%y%m_%d_%H%M%S")
    weight_dir = Path(weight_dir)
    fold_paths = []
    fold_logs = {}

    for fold in range(len(dataset[elec_idx])):
        model.load_state_dict(init_state)  # 每折从初始化重新开始，避免跨折带权重
        optimizer.state.clear()
        schedule.best = float("inf")

        train_dataloader = DataLoader(TensorDataset(dataset[elec_idx][fold]["train_wins"], dataset[elec_idx][fold]["y_train"]), batch_size=batch_size, shuffle=True)
        val_dataloader = DataLoader(TensorDataset(dataset[elec_idx][fold]["val_wins"], dataset[elec_idx][fold]["y_val"]), batch_size=batch_size, shuffle=False)

        patience_counter = 0
        best_val_loss = float("inf")
        best_model_state = None
        fold_logs[f"fold_{fold + 1}"] = {"epochs": []}

        for epoch in range(epochs):
            start_time = time.time()
            train_loss = 0
            val_loss = 0

            model.train()
            for (x, y) in train_dataloader:
                x, y = x.to("cuda"), y.to("cuda")
                optimizer.zero_grad()
                out = model(x)
                loss = loss_function(out, y)
                loss.backward()
                optimizer.step()
                train_loss += loss.item()

            avg_train_loss = train_loss / len(train_dataloader)

            model.eval()
            with torch.no_grad():
                val_targets = []
                val_probs = []
                for (x, y) in val_dataloader:
                    x, y = x.to("cuda"), y.to("cuda")
                    out = model(x)
                    loss = loss_function(out, y)
                    val_loss += loss.item()
                    probs = torch.sigmoid(out)
                    val_probs.extend(probs.cpu().numpy().flatten())
                    val_targets.extend(y.cpu().numpy().flatten())

            avg_val_loss = val_loss / len(val_dataloader)
            val_auc = roc_auc_score(val_targets, val_probs)
            schedule.step(avg_val_loss)
            current_lr = optimizer.param_groups[0]["lr"]

            end_time = time.time()
            fold_logs[f"fold_{fold + 1}"]["epochs"].append({
                "epoch": epoch + 1,
                "train_loss": round(avg_train_loss, 6),
                "val_loss": round(avg_val_loss, 6),
                "val_auc": round(val_auc, 4),
                "time": round(end_time - start_time, 2)
            })

            print(f"Epoch [{epoch+1:02d}/{epochs}] | "
                  f"Train Loss: {avg_train_loss:.2e} | "
                  f"Val Loss: {avg_val_loss:.2e} | "
                  f"Val AUC: {val_auc:.4f} | "
                  f"Time: {end_time - start_time:.1f}s"
                  f"lr: {current_lr:.6e}")

            if avg_val_loss < best_val_loss:
                best_val_loss = avg_val_loss
                patience_counter = 0
                best_model_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            else:
                patience_counter += 1
                if patience_counter == patience:
                    print("Early Stopping")
                    fold_logs[f"fold_{fold + 1}"]["early_stop_epoch"] = epoch + 1
                    break

        if best_model_state is not None:
            path = weight_dir / f"{now}_{elec_names[elec_idx]}_fold{fold+1}.pth"
            fold_paths.append(path)
            torch.save(best_model_state, path)
            fold_logs[f"fold_{fold + 1}"]["weight_path"] = path
            fold_logs[f"fold_{fold + 1}"]["best_val_loss"] = round(best_val_loss, 6)

    return fold_paths, fold_logs

In [3]:
windows,meta = t.preprocess(r"D:\data\dianxian\wang2021\chb22",SPH=2*60,SOP=30*60,interictal_thr=2*60*60,window_length=4,window_overlap=2,elec_names=elec_names)

[边界] chb22 末尾不足整窗: ['chb22_11.edf(丢 1.0s)']
[chb22] 原始窗 55774
[chb22]   label=0 39544
[chb22]   label=1 2366
[chb22]   label=2 96
[chb22]   label=3 13768
[chb22]   excluded(overlap) 22
[chb22]   excluded(sph) 174
[chb22]   excluded(exclusion) 13572
[chb22] usable 42006 / excluded 13768
[chb22] Fp1-F7 窗 55774
[chb22] Fp2-F8 窗 55774
[chb22] T7-P7 窗 55774
[chb22] T8-P8 窗 55774
[chb22] P7-O1 窗 55774
[chb22] P8-O2 窗 55774


In [4]:
model = m.ConvBILSTMClassify()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
# 学习率
schedule = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.5,
    patience=5,
    min_lr=1e-5
)
loss_function = nn.BCEWithLogitsLoss()

In [5]:
models = []
for i in range(len(elec_names)):
    model = m.ConvBILSTMClassify()
    models.append(model)

dataset = t.train_val_test(elec_names, meta, windows, seed, val=(0.1, 0.2), method="LOOCV")

now = datetime.now().strftime("%y%m_%d_%H%M%S")
log_dir = root / "train_log"
combined_manifest = {
    "config": {
        "learning_rate": learning_rate,
        "weight_decay": weight_decay,
        "batch_size": batch_size,
        "dropout": dropout,
        "patience": patience,
        "epochs": epochs,
        "channal_attention": channal_attention,
        "temporal_attention": temporal_attention,
        "hidden_size": hidden_size
    },
    "weights": [],  # 全局只留一份权重列表
    "logs": {}  # 按电极名分别记录详细日志
}
for i, model in enumerate(models):
    print(f"正在训练model：{elec_names[i]}")
    # optimizer = torch.optim.Adam(model.parameters(),lr=learning_rate,weight_decay=weight_decay)
    # loss_function = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    # 学习率
    schedule = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="min",
        factor=0.5,
        patience=5,
        min_lr=1e-5
    )
    loss_function = nn.BCEWithLogitsLoss()
    fold_paths,fold_logs = train_elec_model(model=model,optimizer=optimizer,loss_function=loss_function,schedule=schedule,dataset=dataset,elec_names=elec_names,elec_idx=i,weight_dir=weight,patience=patience,epochs=epochs)

    combined_manifest["weights"].extend(fold_paths)  # 把这一电极的所有权重路径拼进去
    combined_manifest["logs"][elec_names[i]] = fold_logs  # 按电极名存入日志

    del model, optimizer, loss_function
    torch.cuda.empty_cache()
    gc.collect()
    print(f"[清理] {elec_names[i]} 训练完成，显存已释放。")

json_path = log_dir / f"{now}_weights_manifest.json"
with open(json_path, "w", encoding="utf-8") as f:
    json.dump(combined_manifest, f, indent=4, ensure_ascii=False, default=str)

[划分] method=LOOCV 共 3 折
[划分] seizure  train(pre/int) val(pre/int)   test(pre/int)  autoenc(train/val)
[划分] S0       570/570        898/898        898/898        47886/5321      
[划分] S1       570/570        898/898        898/898        47886/5321      
[划分] S2       898/898        898/898        570/570        44646/4961      
[划分] method=LOOCV 共 3 折
[划分] seizure  train(pre/int) val(pre/int)   test(pre/int)  autoenc(train/val)
[划分] S0       570/570        898/898        898/898        47886/5321      
[划分] S1       570/570        898/898        898/898        47886/5321      
[划分] S2       898/898        898/898        570/570        44646/4961      
[划分] method=LOOCV 共 3 折
[划分] seizure  train(pre/int) val(pre/int)   test(pre/int)  autoenc(train/val)
[划分] S0       570/570        898/898        898/898        47886/5321      
[划分] S1       570/570        898/898        898/898        47886/5321      
[划分] S2       898/898        898/898        570/570        44646/4961      
[划分] metho

In [6]:
path = r"C:\Users\hank\Desktop\Research Reproduction\dianxian\self\train_log\2611_13_180841_weights_manifest.json"

fusion_models = t.train_elec_atten(dataset, path=path)
t.test_elec_model(dataset, fusion_models, path=path)

正在训练第0折的融合模型...
正在训练第1折的融合模型...
正在训练第2折的融合模型...
正在测试第0折
📊 分类评估指标 (Threshold = 0.5)
测试集样本数: 1796 (阳性 898 / 阴性 898)
AUC-ROC     : 0.6050
Accuracy    : 0.5947
Balanced Acc: 0.5947
Sensitivity : 0.8764 (Recall)
Specificity : 0.3129
Precision   : 0.5605
F1-Score    : 0.6838
----------------------------------------
混淆矩阵: TP=787, FP=617, TN=281, FN=111
📊 分类评估指标 (Threshold = 0.5)
测试集样本数: 1796 (阳性 898 / 阴性 898)
AUC-ROC     : 0.6147
Accuracy    : 0.5312
Balanced Acc: 0.5312
Sensitivity : 0.8931 (Recall)
Specificity : 0.1693
Precision   : 0.5181
F1-Score    : 0.6558
----------------------------------------
混淆矩阵: TP=802, FP=746, TN=152, FN=96
📊 分类评估指标 (Threshold = 0.5)
测试集样本数: 1796 (阳性 898 / 阴性 898)
AUC-ROC     : 0.7398
Accuracy    : 0.6927
Balanced Acc: 0.6927
Sensitivity : 0.8563 (Recall)
Specificity : 0.5290
Precision   : 0.6451
F1-Score    : 0.7359
----------------------------------------
混淆矩阵: TP=769, FP=423, TN=475, FN=129
📊 分类评估指标 (Threshold = 0.5)
测试集样本数: 1796 (阳性 898 / 阴性 898)
AUC-ROC    